<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/maze_solvers_competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Maze Solving Contest: Algorithm Analysis
**Author:** Mugambi Ndwiga
**Instagram:** @craftsandengineering

This notebook provides a side-by-side comparison of various pathfinding and search algorithms on a randomly generated maze. Each algorithm is visualized to demonstrate its exploration strategy and efficiency in discovering the final path.

## Algorithms Included

1. **A***: Utilizes heuristics to find the shortest path efficiently.
2. **Dijkstra**: Guaranteed shortest path by exploring all directions equally.
3. **Greedy Best-First**: Prioritizes nodes closest to the goal based on a heuristic.
4. **BFS (Breadth-First Search)**: Explores level-by-level, finding the shortest path in unweighted grids.
5. **DFS (Depth-First Search)**: Explores as deep as possible before backtracking.
6. **Bidirectional BFS**: Simultaneously searches from Start and Goal to find a meeting point.
7. **Weighted A***: A variation of A* prioritizing heuristics for faster convergence.
8. **Recursive Backtrack**: A randomized depth-first search approach.
9. **Wall Follower**: A logic-based algorithm that follows the right-hand boundary to reach the exit.

In [1]:
"""
Epic Maze Solving Contest – 9 Algorithms Face Off
Author: Mugambi Ndwiga • Instagram: @craftsandengineering
"""

import numpy as np
import cv2
import heapq
import random
from collections import deque
import time
import os
import io
import base64
from IPython.display import HTML
from google.colab import files
from tqdm.auto import tqdm

# ======================== CONFIGURATION ========================
# MAZE_SIZE determines the logical grid dimensions (cells)
MAZE_SIZE = 20
MAZE_ROWS, MAZE_COLS = MAZE_SIZE, MAZE_SIZE
# VISUAL dimensions account for walls between cells (2n + 1)
VISUAL_H, VISUAL_W = 2 * MAZE_ROWS + 1, 2 * MAZE_COLS + 1
START, GOAL = (0, 0), (MAZE_ROWS - 1, MAZE_COLS - 1)

# Saturated dark colors (BGR) for search trails of each algorithm
ALGO_COLORS = {
    'A*': (120, 0, 0), 'Dijkstra': (0, 0, 120), 'Greedy Best-First': (0, 100, 0),
    'BFS': (120, 60, 0), 'DFS': (60, 0, 120), 'Bidirectional BFS': (60, 60, 60),
    'Weighted A*': (120, 0, 60), 'Recursive Backtrack': (100, 100, 0), 'Wall Follower': (0, 100, 100)
}

FINAL_PATH_COLOR = (0, 255, 0) # Neon Green for the completed path
FPS = 12
SUBPLOT_SIZE = 512 # Size of individual algorithm windows
TITLE_H = 100      # Header height for the video
VIDEO_SIZE = (SUBPLOT_SIZE * 3, SUBPLOT_SIZE * 3 + TITLE_H)

# ======================== MAZE GENERATION ========================
def generate_maze(rows, cols):
    """Generates a perfect maze using Randomized Depth-First Search (Recursive Backtracker)."""
    grid = np.zeros((2*rows+1, 2*cols+1), dtype=np.uint8)
    stack, visited = [START], np.zeros((rows, cols), dtype=bool)
    visited[START] = True
    grid[2*START[0]+1, 2*START[1]+1] = 1 # Mark start cell as path
    dirs = [(-1,0), (1,0), (0,-1), (0,1)]

    while stack:
        r, c = stack[-1]
        # Find unvisited neighbors
        neighs = [(r+dr, c+dc, dr, dc) for dr, dc in dirs if 0 <= r+dr < rows and 0 <= c+dc < cols and not visited[r+dr, c+dc]]
        if neighs:
            nr, nc, dr, dc = random.choice(neighs)
            # Remove wall between current cell and chosen neighbor
            grid[2*r+1+dr, 2*c+1+dc], grid[2*nr+1, 2*nc+1], visited[nr, nc] = 1, 1, True
            stack.append((nr, nc))
        else: stack.pop()
    return grid

def build_valid_moves(grid_v, rows, cols):
    """Pre-calculates valid neighbor moves for each cell to optimize search performance."""
    moves = [[[] for _ in range(cols)] for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
                # Check if the wall is broken at the 2n+1 grid coordinate
                if 0 <= r+dr < rows and 0 <= c+dc < cols and grid_v[2*r+1+dr, 2*c+1+dc] == 1:
                    moves[r][c].append((r+dr, c+dc))
    return moves

# ======================== SOLVERS ========================
class SolverBase:
    """Base class for pathfinding algorithms with path reconstruction logic."""
    def __init__(self, name, start, goal, moves):
        self.name, self.start, self.goal, self.moves = name, start, goal, moves
        self.rows, self.cols = len(moves), len(moves[0])

    def _reconstruct(self, cf):
        """Backtracks from goal to start using the 'came_from' dictionary."""
        p, curr = [], self.goal
        while curr is not None: p.append(curr); curr = cf.get(curr)
        return p[::-1]

class AStarSolver(SolverBase):
    """A* Algorithm: Uses f(n) = g(n) + h(n) where h is Manhattan distance."""
    def solve_generator(self):
        f, cf, cost = [(0, 0, self.start)], {self.start: None}, {self.start: 0}; v, cnt = np.zeros((self.rows, self.cols), bool), 1
        while f:
            _, _, cur = heapq.heappop(f)
            v[cur] = True
            if cur == self.goal: yield {'v': v.copy(), 'p': self._reconstruct(cf)}; return
            yield {'v': v.copy(), 'p': None}
            for n in self.moves[cur[0]][cur[1]]:
                nc = cost[cur] + 1
                if n not in cost or nc < cost[n]:
                    cost[n] = nc; pr = nc + abs(n[0]-self.goal[0]) + abs(n[1]-self.goal[1]);
                    heapq.heappush(f, (pr, cnt, n)); cf[n], cnt = cur, cnt + 1

class DijkstraSolver(SolverBase):
    """Dijkstra's Algorithm: Uniform cost search (equivalent to BFS here)."""
    def solve_generator(self):
        f, cf, cost = [(0, self.start)], {self.start: None}, {self.start: 0}; v = np.zeros((self.rows, self.cols), bool)
        while f:
            c, cur = heapq.heappop(f)
            v[cur] = True
            if cur == self.goal: yield {'v': v.copy(), 'p': self._reconstruct(cf)}; return
            yield {'v': v.copy(), 'p': None}
            for n in self.moves[cur[0]][cur[1]]:
                if n not in cost or c+1 < cost[n]: cost[n] = c+1; heapq.heappush(f, (c+1, n)); cf[n] = cur

class GreedySolver(SolverBase):
    """Greedy Best-First: Prioritizes node with the smallest heuristic value."""
    def solve_generator(self):
        f, cf, v = [(0, random.random(), self.start)], {self.start: None}, np.zeros((self.rows, self.cols), bool)
        while f:
            _, _, cur = heapq.heappop(f)
            v[cur] = True
            if cur == self.goal: yield {'v': v.copy(), 'p': self._reconstruct(cf)}; return
            yield {'v': v.copy(), 'p': None}
            for n in self.moves[cur[0]][cur[1]]:
                if n not in cf:
                    h = abs(n[0]-self.goal[0]) + abs(n[1]-self.goal[1]);
                    heapq.heappush(f, (h, random.random(), n)); cf[n] = cur

class BFSSolver(SolverBase):
    """Breadth-First Search: Explores all neighbors at current depth before moving deeper."""
    def solve_generator(self):
        q, cf, v = deque([self.start]), {self.start: None}, np.zeros((self.rows, self.cols), bool); v[self.start] = True
        while q:
            cur = q.popleft()
            if cur == self.goal: yield {'v': v.copy(), 'p': self._reconstruct(cf)}; return
            yield {'v': v.copy(), 'p': None}
            for n in self.moves[cur[0]][cur[1]]:
                if not v[n]: v[n], cf[n] = True, cur; q.append(n)

class DFSSolver(SolverBase):
    """Depth-First Search: Follows a path as far as possible before backtracking."""
    def solve_generator(self):
        s, cf, v = [self.start], {self.start: None}, np.zeros((self.rows, self.cols), bool)
        while s:
            cur = s.pop()
            if v[cur]: continue
            v[cur] = True
            if cur == self.goal: yield {'v': v.copy(), 'p': self._reconstruct(cf)}; return
            yield {'v': v.copy(), 'p': None}
            for n in self.moves[cur[0]][cur[1]]:
                if not v[n]: cf[n] = cur; s.append(n)

class BiBFSSolver(SolverBase):
    """Bidirectional BFS: Searches from both ends simultaneously to reduce search space."""
    def solve_generator(self):
        qs, qg, cfs, cfg = deque([self.start]), deque([self.goal]), {self.start: None}, {self.goal: None}
        vs, vg = np.zeros((self.rows, self.cols), bool), np.zeros((self.rows, self.cols), bool)
        vs[self.start], vg[self.goal] = True, True
        while qs and qg:
            for q, v_s, v_o, cf_s, cf_o in [(qs, vs, vg, cfs, cfg), (qg, vg, vs, cfg, cfs)]:
                if not q: continue
                cur = q.popleft()
                yield {'v': vs | vg, 'p': None}
                for n in self.moves[cur[0]][cur[1]]:
                    if not v_s[n]:
                        v_s[n], cf_s[n] = True, cur; q.append(n)
                        if v_o[n]: # Search frontiers met
                            p1, p2, c = [], [], n
                            while c is not None: p1.append(c); c = cfs.get(c)
                            c = cfg.get(n)
                            while c is not None: p2.append(c); c = cfg.get(c)
                            yield {'v': vs | vg, 'p': p1[::-1] + p2}; return

class WeightedAStar(SolverBase):
    """Weighted A*: Over-emphasizes heuristic (W > 1) for faster, potentially non-optimal search."""
    def solve_generator(self):
        W = 3.0; f, cf, cost = [(0, self.start)], {self.start: None}, {self.start: 0}; v = np.zeros((self.rows, self.cols), bool)
        while f:
            _, cur = heapq.heappop(f)
            v[cur] = True
            if cur == self.goal: yield {'v': v.copy(), 'p': self._reconstruct(cf)}; return
            yield {'v': v.copy(), 'p': None}
            for n in self.moves[cur[0]][cur[1]]:
                nc = cost[cur] + 1
                if n not in cost or nc < cost[n]:
                    cost[n] = nc; h = abs(n[0]-self.goal[0]) + abs(n[1]-self.goal[1]);
                    heapq.heappush(f, (nc + W*h, n)); cf[n] = cur

class RecursiveBacktrack(SolverBase):
    """Recursive Backtrack (DFS variant): Randomized exploration with explicit backtracking."""
    def solve_generator(self):
        v, p, cf = np.zeros((self.rows, self.cols), bool), [self.start], {self.start: None}
        while p:
            cur = p[-1]; v[cur] = True
            if cur == self.goal: yield {'v': v.copy(), 'p': self._reconstruct(cf)}; return
            yield {'v': v.copy(), 'p': None}
            unvisited = [n for n in self.moves[cur[0]][cur[1]] if not v[n]]
            if unvisited: nxt = random.choice(unvisited); cf[nxt] = cur; p.append(nxt)
            else: p.pop()

class WallFollower(SolverBase):
    """Wall Follower: Simple reactive logic following the right-hand wall boundary."""
    def solve_generator(self):
        cur, v, p = self.start, np.zeros((self.rows, self.cols), bool), [self.start]; facing = (0, 1); dirs = [(0,1), (1,0), (0,-1), (-1,0)]
        while cur != self.goal:
            v[cur] = True; yield {'v': v.copy(), 'p': None}
            idx = dirs.index(facing)
            # Attempt to turn right, then straight, then left, then back
            for i in range(-1, 3):
                d = dirs[(idx + i) % 4]; nxt = (cur[0]+d[0], cur[1]+d[1])
                if nxt in self.moves[cur[0]][cur[1]]: cur, facing = nxt, d; p.append(cur); break
            if len(p) > 2500: break # Safety break
        v[cur] = True; yield {'v': v.copy(), 'p': p}

# ======================== UTILITIES ========================
def display_video(file_path):
    """Encodes and embeds the MP4 video in the notebook output."""
    video = io.open(file_path, 'r+b').read(); encoded = base64.b64encode(video)
    return HTML(data='''<video controls width="800"><source src="data:video/mp4;base64,{0}" type="video/mp4" /></video>'''.format(encoded.decode('ascii')))

def render_video(snaps_dict, grid_vis, rankings_final):
    """Renders a multi-grid visualization into an MP4 file with rankings and watermarks."""
    fname = 'maze_showdown_mugambi.mp4'; out = cv2.VideoWriter(fname, cv2.VideoWriter_fourcc(*'mp4v'), FPS, VIDEO_SIZE)
    max_f = max(len(s) for s in snaps_dict.values())
    base_p = (grid_vis == 1)

    # Main animation loop
    last_img = None
    for step in tqdm(range(max_f), desc="Rendering"):
        img = np.zeros((VIDEO_SIZE[1], VIDEO_SIZE[0], 3), dtype=np.uint8)
        for i, name in enumerate(ALGO_COLORS.keys()):
            snaps = snaps_dict.get(name, [])
            state = snaps[min(step, len(snaps)-1)]
            sub = np.zeros((VISUAL_H, VISUAL_W, 3), np.uint8); sub[base_p] = (255,255,255)
            v = state['v']
            if v.any():
                rows_v, cols_v = v.nonzero()
                sub[2*rows_v+1, 2*cols_v+1] = ALGO_COLORS[name]
            res = cv2.resize(sub, (SUBPLOT_SIZE, SUBPLOT_SIZE), interpolation=cv2.INTER_NEAREST)
            found_p = state.get('p')
            rank_text = ""
            if found_p is not None and step >= len(snaps) - 1:
                rank_text = f" #{rankings_final[name]}"
                pts = np.array([(2*c+1, 2*r+1) for r, c in found_p], np.float32)
                pts *= (SUBPLOT_SIZE / VISUAL_W); pts = pts.astype(np.int32)
                cv2.polylines(res, [pts], False, FINAL_PATH_COLOR, 14, cv2.LINE_AA)
            cv2.rectangle(res, (0,0), (SUBPLOT_SIZE, 40), (0,0,0), -1)
            cv2.putText(res, name + rank_text, (15, 30), 0, 0.9, (255,255,255), 2)
            r_idx, c_idx = i // 3, i % 3
            img[TITLE_H + r_idx*SUBPLOT_SIZE : TITLE_H + (r_idx+1)*SUBPLOT_SIZE, c_idx*SUBPLOT_SIZE : (c_idx+1)*SUBPLOT_SIZE] = res
        cv2.putText(img, "@craftsandengineering", (VIDEO_SIZE[0] - 320, VIDEO_SIZE[1] - 20), 0, 0.7, (150,150,150), 2)
        out.write(img)
        last_img = img

    # Add a 3 second buffer of the final state
    for _ in range(FPS * 3): out.write(last_img)

    # Add a title card at the end
    title_card = np.zeros((VIDEO_SIZE[1], VIDEO_SIZE[0], 3), dtype=np.uint8)
    cv2.putText(title_card, "FINAL RANKINGS", (VIDEO_SIZE[0]//2 - 200, 150), 0, 1.8, (255, 255, 255), 4)
    sorted_ranks = sorted(rankings_final.items(), key=lambda x: x[1])
    for i, (name, rank) in enumerate(sorted_ranks):
        txt = f"#{rank} {name}"
        cv2.putText(title_card, txt, (VIDEO_SIZE[0]//2 - 150, 250 + i*50), 0, 1.1, (200, 200, 200), 2)

    cv2.putText(title_card, "Author: Mugambi Ndwiga", (VIDEO_SIZE[0]//2 - 180, VIDEO_SIZE[1] - 150), 0, 1.0, (150, 255, 150), 2)
    cv2.putText(title_card, "Instagram: @craftsandengineering", (VIDEO_SIZE[0]//2 - 240, VIDEO_SIZE[1] - 100), 0, 1.0, (150, 150, 255), 2)

    for _ in range(FPS * 5): out.write(title_card)

    out.release(); return fname

if __name__ == "__main__":
    # 1. Initialize Maze and Pre-calculate moves
    grid = generate_maze(MAZE_ROWS, MAZE_COLS); mv = build_valid_moves(grid, MAZE_ROWS, MAZE_COLS)
    solvers = [(AStarSolver, 'A*'), (DijkstraSolver, 'Dijkstra'), (GreedySolver, 'Greedy Best-First'), (BFSSolver, 'BFS'), (DFSSolver, 'DFS'), (BiBFSSolver, 'Bidirectional BFS'), (WeightedAStar, 'Weighted A*'), (RecursiveBacktrack, 'Recursive Backtrack'), (WallFollower, 'Wall Follower')]
    results, solve_times = {}, []

    # 2. Run Solvers and capture search snapshots
    print("Solving...")
    for cls, name in tqdm(solvers):
        full_gen = list(cls(name, START, GOAL, mv).solve_generator())
        solve_times.append((len(full_gen), name))
        sampled = full_gen[::2] # Sample frames to speed up video rendering
        if len(full_gen) > 0 and (len(sampled) == 0 or sampled[-1].get('p') is None):
             sampled.append(full_gen[-1])
        results[name] = sampled

    # 3. Calculate rankings based on search exploration speed
    solve_times.sort(); rankings_final = {name: i+1 for i, (_, name) in enumerate(solve_times)}

    # 4. Render and display result
    fname = render_video(results, grid, rankings_final)
    # display(display_video(fname))
    files.download(fname)

Solving...


  0%|          | 0/9 [00:00<?, ?it/s]

Rendering:   0%|          | 0/264 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>